# Faruq-v3 — Circle-CPE Matched Objective Screening (Kaggle)

Kaggle GPU version of the frozen seed42 validation-only Circle-CPE screen.

**Protocol unchanged:** CIR0 is matched to CPE0; CIR7 is matched to CPE7. Same D0 parent, 128-D P3/P4/P5 training-only projection, assignment screen, schedule, and native inference. Only SupCon is replaced by Circle-style pair weighting.

Frozen before results: `margin=0.25`, `gamma=256`, `lambda=0.005`. No post-result tuning. **Test must never be present/extracted/opened.**

Attach one Kaggle Dataset containing these five inputs (flat renamed files or original project folder structure):
- `faruq-development-v3-grouped.tar`
- `D0_seed42_best.pt` OR `.../D0_seed42/weights/best.pt`
- `D0FT_seed42_val.json`
- `CPE0_seed42_val.json`
- `CPE7_seed42_val.json`


In [ ]:
import os, shutil, subprocess, sys, tarfile, time, json
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
REPO = WORK_ROOT / 'coffee-bean-detection'
BRANCH = 'agent/circle-cpe-screening'

assert INPUT_ROOT.exists(), '/kaggle/input tidak ditemukan.'
os.chdir(WORK_ROOT)
if REPO.exists():
    shutil.rmtree(REPO)

clone = ['git','clone','--depth','1','--branch',BRANCH,
         'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(3):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 2:
        raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)

subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
sys.path.insert(0, str(REPO/'src'))
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())


In [ ]:
import torch
assert torch.cuda.is_available(), 'Aktifkan Accelerator GPU pada Kaggle Notebook.'
print('GPU:', torch.cuda.get_device_name(0))
print('INPUT DATASETS:')
for p in sorted(INPUT_ROOT.iterdir()):
    print(' -', p)


In [ ]:
def unique_by_exact_name(name: str):
    matches = [p for p in INPUT_ROOT.rglob(name) if p.is_file()]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f'WARNING: multiple {name}:')
        for p in matches:
            print(' ', p)
    return None

def resolve_artifact(label, preferred_names=(), path_suffixes=()):
    for name in preferred_names:
        p = unique_by_exact_name(name)
        if p is not None:
            print(label, '->', p)
            return p

    candidates = []
    for p in INPUT_ROOT.rglob('*'):
        if not p.is_file():
            continue
        norm = str(p).replace('\\','/')
        if any(norm.endswith(suffix) for suffix in path_suffixes):
            candidates.append(p)

    if len(candidates) == 1:
        print(label, '->', candidates[0])
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f'{label} tidak ditemukan. Expected names={preferred_names} suffixes={path_suffixes}'
        )
    raise RuntimeError(f'{label} ambigu: {candidates}')

ARCHIVE = resolve_artifact(
    'ARCHIVE',
    preferred_names=('faruq-development-v3-grouped.tar',),
    path_suffixes=('/bundles/faruq-development-v3-grouped.tar',),
)
D0_CHECKPOINT = resolve_artifact(
    'D0',
    preferred_names=('D0_seed42_best.pt',),
    path_suffixes=('/experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
                   '/D0_seed42/weights/best.pt'),
)
D0FT_REPORT = resolve_artifact(
    'D0FT',
    preferred_names=('D0FT_seed42_val.json',),
    path_suffixes=('/experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json',),
)
CPE0_REPORT = resolve_artifact(
    'CPE0',
    preferred_names=('CPE0_seed42_val.json',),
    path_suffixes=('/experiments/faruq-v3-fsce-cpe-screening-v1/val_reports/CPE0_seed42_val.json',),
)
CPE7_REPORT = resolve_artifact(
    'CPE7',
    preferred_names=('CPE7_seed42_val.json',),
    path_suffixes=('/experiments/faruq-v3-fsce-cpe-screening-v1/val_reports/CPE7_seed42_val.json',),
)


In [ ]:
DATA_ROOT = WORK_ROOT / 'faruq-development-v3-grouped'
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'

if DATA_ROOT.exists() and not GROUPED_SUMMARY.is_file():
    shutil.rmtree(DATA_ROOT)

if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall(WORK_ROOT, filter='data')

assert GROUPED_SUMMARY.is_file(), f'Grouped summary tidak ditemukan: {GROUPED_SUMMARY}'
assert not (DATA_ROOT/'test').exists(), 'STOP: test split terdeteksi. Jangan lanjut.'

OUTPUT_ROOT = WORK_ROOT / 'experiments/faruq-v3-circle-cpe-screening-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('TEST PRESENT:', (DATA_ROOT/'test').exists())


In [ ]:
command = [sys.executable,'-m','pytest','-q','tests/test_circle_cpe.py']
print('STATIC CHECK:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
from coffee_detector.circle_cpe import circle_pair_loss
from coffee_detector.fsce_cpe.loss import cpe_supervised_contrastive_loss

g = torch.Generator().manual_seed(42)
z = torch.randn(200,128,generator=g)
y = torch.arange(200) % 21

sup = cpe_supervised_contrastive_loss(z,y,temperature=0.2).item()
cir = circle_pair_loss(z,y,margin=0.25,gamma=256.0).item()
print({
    'raw_supcon': sup,
    'weighted_supcon_lambda_0.5': 0.5*sup,
    'raw_circle': cir,
    'weighted_circle_lambda_0.005': 0.005*cir,
})


In [ ]:
command = [
    sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_circle_cpe_screening',
    '--data-root',str(DATA_ROOT),
    '--grouped-summary',str(GROUPED_SUMMARY),
    '--d0-checkpoint',str(D0_CHECKPOINT),
    '--d0ft-report',str(D0FT_REPORT),
    '--cpe0-report',str(CPE0_REPORT),
    '--cpe7-report',str(CPE7_REPORT),
    '--output-root',str(OUTPUT_ROOT),
    '--seed','42','--device','0','--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.run(command, cwd=REPO, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(process.stdout)
if process.returncode:
    raise RuntimeError(f'Circle-CPE gagal: {process.returncode}')


In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/circle_cpe_seed42_screening.json'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))

assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
assert result['test_opened'] is False

rows = []
for name in ('D0FT','CPE0','CPE7'):
    rows.append({'model':name, **result['controls'][name]})
for name, metrics in result['candidate'].items():
    rows.append({'model':name, **metrics})

display(pd.DataFrame(rows).style.format({
    k:'{:.2%}' for k in (
        'macro_map50_95',
        'bottom3_class_map50_95',
        'worst_class_map50_95',
    )
}))

for arm in ('CIR0','CIR7'):
    d = result['decisions'][arm]
    print('\n', arm, d['decision'])
    print('matched control:', d['matched_supcon_control'])
    print('delta vs matched CPE:',
          {k:f'{v*100:+.2f} pp' for k,v in d['delta_vs_matched_cpe'].items()})
    print('delta vs D0FT:',
          {k:f'{v*100:+.2f} pp' for k,v in d['delta_vs_D0FT'].items()})
    print('criteria:', d['criteria'])

print('\nRETAINED:', result['retained_for_multiseed_confirmation'])
print('NEXT:', result['next_action'])
print('SUMMARY:', SUMMARY)


In [ ]:
ZIP_BASE = WORK_ROOT / 'faruq-v3-circle-cpe-screening-v1'
zip_path = shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=OUTPUT_ROOT)
print('KAGGLE OUTPUT ZIP:', zip_path)
print('Use Save Version agar /kaggle/working output tersimpan.')
print('Kirim tabel + decision CIR0/CIR7. Jangan membuka test.')
